In [1]:
import ee

# REPLACE THIS with the Personal ID you just found/created
my_personal_project = 'sensiblesat' 

try:
    ee.Initialize(project=my_personal_project)
    print(f"Success! Connected to personal project: {my_personal_project}")
except Exception as e:
    print("New auth needed for this project...")
    # This might pop up the browser one last time to link THIS project
    ee.Authenticate()
    ee.Initialize(project=my_personal_project)

Success! Connected to personal project: sensiblesat


In [68]:
import ee
import geemap

# REPLACE THIS with the Personal ID you just found/created
my_personal_project = 'sensiblesat' 

# 2. Define the Region of Interest (Canary Islands)
# Buffer 30km (30,000m) to cover the whole island
roi = ee.Geometry.Point([-15.5474, 27.9202]).buffer(30000)

# 3. Load Sentinel-2 Image (Cloud-Free Mosaic logic)
collection = ee.ImageCollection('COPERNICUS/S2_HARMONIZED') \
    .filterBounds(roi) \
    .filterDate('2023-01-01', '2023-02-01') \
    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20)) \
    .median() # The "Pancake Stack" trick

# 4. Create the Map
Map = geemap.Map()
Map.add_basemap('Stadia.StamenTerrain')

Map.centerObject(roi, 10)

# 5. Add the Layer (True Color: Red, Green, Blue)
vis_params = {'min': 0, 'max': 3000, 'bands': ['B4', 'B3', 'B2']}
Map.addLayer(collection, vis_params, "Sentinel-2 Tenerife")

# 6. Display
Map

Map(center=[27.920230776216357, -15.547399034103881], controls=(WidgetControl(options=['position', 'transparen…

In [54]:
import ee
import geemap

# 2. Define Location: London (A classic urban jungle)
# We create a point and buffer it to get a 20km circle
# Buffer the point by 30km to create a visible circle
# Buffer 30km (30,000m) to cover the whole island
roi = ee.Geometry.Point([-15.5474, 27.9202]).buffer(35000)

# 3. Load Dynamic World & Filter
# We filter for just ONE month to get a clean snapshot
dw_collection = ee.ImageCollection("GOOGLE/DYNAMICWORLD/V1") \
    .filterBounds(roi) \
    .filterDate('2015-01-01', '2015-12-31')

# 4. The "Pancake Stack" Logic (Reducer)
# We select the 'built' band (Probability 0 to 1)
# We take the mean() to average out any cloud noise or transient glitches
urban_occupancy = dw_collection.select('built').mean().clip(roi)

# 5. Create the Map
Map = geemap.Map()
Map.centerObject(roi, 11)

# 6. Visualization: The "Heatmap" Style
# 0 (Black) = No buildings (Parks, River Thames)
# 1 (White/Yellow) = Dense Concrete (City Center)
urban_vis = {
    'min': 0,
    'max': 0.8, # Cap at 0.8 to make the city pop brighter
    'palette': ['000000', '2c0e38', 'b3204d', 'ffc636', 'ffffff'] # Magma-like colors
}

# 7. Add Layers
# Add the satellite image underneath for reference
Map.addLayer(ee.ImageCollection('COPERNICUS/S2_HARMONIZED')
             .filterBounds(roi)
             .filterDate('2025-12-01', '2026-01-01')
             .median().clip(roi), 
             {'min':0, 'max':3000, 'bands':['B4','B3','B2']}, 'True Color Base')

Map.addLayer(urban_occupancy, urban_vis, 'Urban Probability (Occupancy)')

Map

Map(center=[27.920235905394637, -15.547398873127054], controls=(WidgetControl(options=['position', 'transparen…

In [77]:
import ee
import geemap

# 2. Define Location: Delhi, India
# A 40km buffer captures the most intense expansion zones
roi = ee.Geometry.Point([-15.5474, 27.9202]).buffer(35000)

# 3. Fetch Dynamic World Data
# Using 'max' for the early dates helps fill gaps from lower satellite coverage
built_2015 = ee.ImageCollection("GOOGLE/DYNAMICWORLD/V1") \
    .filterBounds(roi) \
    .filterDate('2016-01-01', '2016-12-13') \
    .select('built').median().clip(roi)

built_2026 = ee.ImageCollection("GOOGLE/DYNAMICWORLD/V1") \
    .filterBounds(roi) \
    .filterDate('2025-06-01', '2026-01-01') \
    .select('built').median().clip(roi)

# 4. QUANTIFICATION LOGIC (The Math)
threshold = 0.5  # 50% probability counts as "Urban"

# Create binary masks
mask_2015 = built_2015.gt(threshold)
mask_2026 = built_2026.gt(threshold)

# 5. VISUALIZATION
Map = geemap.Map()

Map.add_basemap('Esri.WorldGrayCanvas')

urban_vis = {
    'min': 0, 
    'max': 0.8, 
    'palette': ['000000', '2c0e38', 'b3204d', 'ffc636', 'ffffff']
}

# Create TileLayers for the Split Map
left_layer = geemap.ee_tile_layer(built_2015, urban_vis, 'Urban 2015')
right_layer = geemap.ee_tile_layer(built_2026, urban_vis, 'Urban 2026')

# Add Split Map and UI Elements
Map.split_map(left_layer=left_layer, right_layer=right_layer)
Map.add_text("Left: 2015 | Right: 2026", position='bottomleft')
Map.centerObject(roi, 10)

km2_2015 = get_area(mask_2015, roi)
km2_2026 = get_area(mask_2026, roi)
Map

Map(center=[27.920235905394637, -15.547398873127054], controls=(ZoomControl(options=['position', 'zoom_in_text…

In [64]:
# Calculate Area in Square Kilometers
def get_area(img, region):
    area_px = img.multiply(ee.Image.pixelArea())
    stats = area_px.reduceRegion(
        reducer=ee.Reducer.sum(),
        geometry=region,
        scale=10,
        maxPixels=1e9
    )
    return ee.Number(stats.get('built')).divide(1e6)


km2_2015 = get_area(mask_2015, roi)
km2_2026 = get_area(mask_2026, roi)

growth = km2_2026.subtract(km2_2015)

# Print stats to the console
print(f"--- Urban Expansion Analysis (Canary Islands) ---")
print(f"Urban Area 2015: {km2_2015.getInfo():.2f} km²")
print(f"Urban Area 2026: {km2_2026.getInfo():.2f} km²")
print(f"Net Expansion:    {growth.getInfo():.2f} km²")

# Identify "New" Growth (Red layer)
new_growth = mask_2026.And(mask_2015.Not())
Map.addLayer(new_growth.selfMask(), {'palette': 'FF0000'}, 'Newly Built (2015-2026)')


--- Urban Expansion Analysis (Canary Islands) ---
Urban Area 2015: 197.82 km²
Urban Area 2026: 206.92 km²
Net Expansion:    9.11 km²


In [66]:
def get_area_30(mask, region):
    area_px = mask.multiply(ee.Image.pixelArea())
    stats = area_px.reduceRegion(
        reducer=ee.Reducer.sum(),
        geometry=region,
        scale=30,  # <--- CHANGED FROM 10 TO 30
        maxPixels=1e10,
        bestEffort=True # <--- Adds a safety valve to prevent timeouts
    )
    return ee.Number(stats.get('built')).divide(1e6)

km2_2015 = get_area_30(mask_2015, roi)
km2_2026 = get_area_30(mask_2026, roi)

growth = km2_2026.subtract(km2_2015)

# Print stats to the console
print(f"--- Urban Expansion Analysis (Canary Islands) ---")
print(f"Urban Area 2015: {km2_2015.getInfo():.2f} km²")
print(f"Urban Area 2026: {km2_2026.getInfo():.2f} km²")
print(f"Net Expansion:    {growth.getInfo():.2f} km²")

# Identify "New" Growth (Red layer)
new_growth = mask_2026.And(mask_2015.Not())
Map.addLayer(new_growth.selfMask(), {'palette': 'FF0000'}, 'Newly Built (2015-2026)')


--- Urban Expansion Analysis (Canary Islands) ---
Urban Area 2015: 196.99 km²
Urban Area 2026: 206.19 km²
Net Expansion:    9.20 km²


In [67]:
Map

Map(bottom=439485.0, center=[27.966052996481704, -15.46051025390625], controls=(ZoomControl(options=['position…

In [76]:
Map = geemap.Map()

roi = ee.Geometry.Point([-15.5474, 27.9202]).buffer(35000)

Map.add_basemap('Esri.WorldGrayCanvas')

Map.centerObject(roi, 10)

Map

Map(center=[27.920235905394637, -15.547398873127054], controls=(WidgetControl(options=['position', 'transparen…

In [4]:
import ee

first_image = ee.ImageCollection("GOOGLE/DYNAMICWORLD/V1").first()
print(first_image.propertyNames().getInfo())

['system:version', 'system:id', 'system:time_start', 'dynamicworld_algorithm_version', 'qa_algorithm_version', 'system:footprint', 'system:asset_size', 'system:index', 'system:bands', 'system:band_names']


In [12]:
import ee
import geemap
from ipywidgets import HTML

# 1. Initialize
my_personal_project = 'sensiblesat' 
try:
    ee.Initialize(project=my_personal_project)
except Exception as e:
    ee.Authenticate()
    ee.Initialize(project=my_personal_project)

# 2. Define Location: Gran Canaria
roi = ee.Geometry.Point([-15.5474, 27.9202]).buffer(30000)

# 3. Clean Layer Function (Median + Water Mask)
def get_clean_layer(start_date, end_date):
    # Dynamic World is pre-filtered for clouds < 35%
    collection = ee.ImageCollection("GOOGLE/DYNAMICWORLD/V1") \
        .filterBounds(roi) \
        .filterDate(start_date, end_date)
    
    # We use median to ignore outliers like sunglint or temporary sand
    image = collection.select(['built', 'water']).median().clip(roi)
    
    built = image.select('built')
    water = image.select('water')
    
    # Mask out water where probability is > 0.5 to stop the "Red Sea"
    return built.updateMask(water.lt(0.5))

# Generate layers (using 6-month windows for a cleaner mosaic)
built_2016 = get_clean_layer('2016-01-01', '2016-07-01')
built_2026 = get_clean_layer('2025-06-01', '2026-02-01')

# 4. Identification of Growth
threshold = 0.5 # Confidence threshold as recommended by documentation
mask_2016 = built_2016.gt(threshold)
mask_2026 = built_2026.gt(threshold)
new_growth = mask_2026.And(mask_2016.Not())

# 5. Build the Visual Map
Map = geemap.Map()
Map.clear_layers() # Removes the default OpenStreetMap
Map.add_basemap('Stadia.StamenTerrain') # Adds clean mountains/labels

# Visual parameters for the "Urban Glow"
urban_vis = {
    'min': 0, 
    'max': 0.8, 
    'palette': ['000000', '2c0e38', 'b3204d', 'ffc636', 'ffffff']
}

# Create the TileLayers
left_layer = geemap.ee_tile_layer(built_2016, urban_vis, 'Urban 2016')
right_layer = geemap.ee_tile_layer(built_2026, urban_vis, 'Urban 2026')

# Create a Red mask for New Growth
growth_layer = geemap.ee_tile_layer(new_growth.selfMask(), {'palette': 'FF0000'}, 'New Construction')

# 6. ADD ON-MAP LABELS
# Create HTML labels for the UI
label_2016 = HTML('<div style="color: white; background: rgba(0,0,0,0.5); padding: 5px; border-radius: 3px;"><b>Left: 2016</b></div>')
label_2026 = HTML('<div style="color: white; background: rgba(0,0,0,0.5); padding: 5px; border-radius: 3px;"><b>Right: 2026</b></div>')

# Place labels in the bottom corners
Map.add_text("Left: 2016 | Right: 2026", position='bottomleft')

# 7. Final Map Assembly
Map.split_map(left_layer=left_layer, right_layer=right_layer)
Map.addLayer(growth_layer) # Add the red expansion layer on top of everything
Map.centerObject(roi, 11)

Map

Map(center=[27.920230776216357, -15.547399034103881], controls=(ZoomControl(options=['position', 'zoom_in_text…

In [13]:
# 1. Identification (Thresholding at 0.5 per docs)
threshold = 0.5
mask_2016 = built_2016.gt(threshold)
mask_2026 = built_2026.gt(threshold)

# 2. Category Logic
# STABLE: Urban in both years
stable = mask_2016.And(mask_2026)

# NEW: Only in 2026 (Expansion)
new_growth = mask_2026.And(mask_2016.Not())

# REMOVED: Only in 2016 (Demolition or AI Flicker)
removed = mask_2016.And(mask_2026.Not())

# 3. Visualization
Map = geemap.Map()
Map.clear_layers()
Map.add_basemap('Stadia.StamenTerrain')

# Add layers with distinct colors
Map.addLayer(stable.selfMask(), {'palette': 'FFFFFF'}, 'Stable Urban (White)')
Map.addLayer(new_growth.selfMask(), {'palette': '00FF00'}, 'New Growth (Green)')
Map.addLayer(removed.selfMask(), {'palette': 'FF0000'}, 'Removed/Flicker (Red)')

Map.add_text("White: Stable | Green: New | Red: Removed", position='bottomleft')
Map.centerObject(roi, 12)
Map

Map(center=[27.920230776216357, -15.547399034103881], controls=(WidgetControl(options=['position', 'transparen…

In [14]:
# 1. Define the Area Function (Scale 30 for speed)
def calculate_category_area(mask, name):
    # Multiply the binary mask (0 or 1) by the physical area of each pixel
    area_image = mask.multiply(ee.Image.pixelArea())
    
    stats = area_image.reduceRegion(
        reducer=ee.Reducer.sum(),
        geometry=roi,
        scale=30,
        maxPixels=1e10
    )
    
    # Extract result and convert from m2 to km2
    # Note: 'built' is the band name inherited from the original image
    area_km2 = ee.Number(stats.get('built')).divide(1e6)
    return area_km2

# 2. Run the math for each category
print("Computing island statistics... please wait.")

area_stable  = calculate_category_area(stable, "Stable")
area_new     = calculate_category_area(new_growth, "New Growth")
area_removed = calculate_category_area(removed, "Loss/Flicker")

# 3. Print the Results
print(f"--- Urban Lifecycle Stats (km²) ---")
print(f"✅ Stable (Existing in both): {area_stable.getInfo():.2f} km²")
print(f"📈 New Growth (Expansion):    {area_new.getInfo():.2f} km²")
print(f"📉 Loss/Flicker (Disappeared): {area_removed.getInfo():.2f} km²")

# Calculate Net Change
net_change = area_new.subtract(area_removed)
print(f"Total Net Change:             {net_change.getInfo():.2f} km²")

Computing island statistics... please wait.
--- Urban Lifecycle Stats (km²) ---
✅ Stable (Existing in both): 176.68 km²
📈 New Growth (Expansion):    28.47 km²
📉 Loss/Flicker (Disappeared): 26.98 km²
Total Net Change:             1.50 km²


In [3]:
import ee
import geemap

# 1. Initialize
my_personal_project = 'sensiblesat' 
try:
    ee.Initialize(project=my_personal_project)
except Exception as e:
    ee.Authenticate()
    ee.Initialize(project=my_personal_project)

# 2. Define Location: Gran Canaria
roi = ee.Geometry.Point([-15.5474, 27.9202]).buffer(30000)

# 3. Clean Layer Function (Using MODE and LABELS)
def get_stable_label_layer(start_date, end_date):
    # Load collection
    collection = ee.ImageCollection("GOOGLE/DYNAMICWORLD/V1") \
        .filterBounds(roi) \
        .filterDate(start_date, end_date)
    
    # Use .mode() to find the most frequent label for each pixel
    # This acts as a temporal filter to remove "flicker"
    mode_image = collection.select('label').mode().clip(roi)
    
    # Class 6 in Dynamic World is "Built"
    built_mask = mode_image.eq(6)
    
    # Still apply a water mask as a safety measure for coastal areas
    # Class 0 in Dynamic World is "Water"
    water_mask = mode_image.eq(0)
    
    return built_mask.updateMask(water_mask.Not())

# Generate the stable masks for both years
# Using a full year for 2016 and the available 2025/26 data for the latest
mask_2016 = get_stable_label_layer('2016-01-01', '2016-12-31')
mask_2026 = get_stable_label_layer('2025-01-01', '2026-02-01')

# 4. Identification of Growth Categories
stable = mask_2016.And(mask_2026)
new_growth = mask_2026.And(mask_2016.Not())
removed = mask_2016.And(mask_2026.Not())

# 5. Quantification (Scale 30 for speed)
def get_km2(mask):
    area_px = mask.multiply(ee.Image.pixelArea())
    stats = area_px.reduceRegion(
        reducer=ee.Reducer.sum(),
        geometry=roi,
        scale=30,
        maxPixels=1e10
    )
    # The band name in a mode() result usually defaults to 'label'
    return ee.Number(stats.get('label')).divide(1e6)

print("Calculating stable statistics using Label Mode...")
area_stable = get_km2(stable).getInfo()
area_new = get_km2(new_growth).getInfo()
area_loss = get_km2(removed).getInfo()

print(f"--- Results (Label Mode) ---")
print(f"Stable Urban:  {area_stable:.2f} km²")
print(f"New Growth:    {area_new:.2f} km²")
print(f"Loss/Flicker:  {area_loss:.2f} km²")

# 6. Visualization
Map = geemap.Map()
Map.clear_layers()
Map.add_basemap('Stadia.StamenTerrain')

# Add the categories
Map.addLayer(stable.selfMask(), {'palette': 'FFFFFF'}, 'Stable (White)')
Map.addLayer(new_growth.selfMask(), {'palette': '00FF00'}, 'New Growth (Green)')
Map.addLayer(removed.selfMask(), {'palette': 'FF0000'}, 'Loss/Flicker (Red)')

Map.add_text("White: Stable | Green: New | Red: Loss", position='bottomleft')
Map.centerObject(roi, 11)
Map

Calculating stable statistics using Label Mode...


KeyboardInterrupt: 

In [2]:
import ee
import geemap

# 1. Initialize
my_personal_project = 'sensiblesat' 
try:
    ee.Initialize(project=my_personal_project)
except Exception as e:
    ee.Authenticate()
    ee.Initialize(project=my_personal_project)

# 2. Define Location: Delhi, India
# A 40km buffer covers the main NCR (National Capital Region)
roi = ee.Geometry.Point([77.2090, 28.6139]).buffer(40000)

# 3. Clean Layer Function (Using Mode for Stability)
def get_stable_label_layer(start_date, end_date):
    # Dynamic World collection is pre-filtered for clouds < 35%
    collection = ee.ImageCollection("GOOGLE/DYNAMICWORLD/V1") \
        .filterBounds(roi) \
        .filterDate(start_date, end_date)
    
    # .mode() finds the most frequent 'Top-1' label for each pixel
    mode_image = collection.select('label').mode().clip(roi)
    
    # Class 6 is "Built"
    built_mask = mode_image.eq(6)
    
    # Class 0 is "Water" (Safety mask)
    water_mask = mode_image.eq(0)
    
    return built_mask.updateMask(water_mask.Not())

# Generate masks for 2016 and 2026
mask_2016 = get_stable_label_layer('2016-01-01', '2016-02-01')
mask_2026 = get_stable_label_layer('2025-01-01', '2026-02-01')

# 4. Identification of Growth Categories
stable = mask_2016.And(mask_2026)
new_growth = mask_2026.And(mask_2016.Not())
removed = mask_2016.And(mask_2026.Not())

# 5. Stats Calculation
def get_km2(mask):
    area_px = mask.multiply(ee.Image.pixelArea())
    stats = area_px.reduceRegion(
        reducer=ee.Reducer.sum(),
        geometry=roi,
        scale=30, # 30m scale is fast for large cities
        maxPixels=1e10
    )
    return ee.Number(stats.get('label')).divide(1e6)

print("Computing Delhi Urban Statistics...")
area_stable = get_km2(stable).getInfo()
area_new = get_km2(new_growth).getInfo()
area_loss = get_km2(removed).getInfo()

print(f"--- Delhi Results (Label Mode) ---")
print(f"✅ Stable Urban:  {area_stable:.2f} km²")
print(f"📈 New Growth:    {area_new:.2f} km²")
print(f"📉 Loss/Flicker:  {area_loss:.2f} km²")
print(f"🚀 Net Expansion: {area_new - area_loss:.2f} km²")

# 6. Visualization
Map = geemap.Map()
Map.clear_layers()
Map.add_basemap('Stadia.StamenTerrain')

Map.addLayer(stable.selfMask(), {'palette': 'FFFFFF'}, 'Stable Urban (White)')
Map.addLayer(new_growth.selfMask(), {'palette': '00FF00'}, 'New Growth (Green)')
Map.addLayer(removed.selfMask(), {'palette': 'FF0000'}, 'Loss/Flicker (Red)')

Map.add_text("Delhi: White (Stable), Green (New), Red (Loss)", position='bottomleft')
Map.centerObject(roi, 10)
Map

Computing Delhi Urban Statistics...


KeyboardInterrupt: 